In [2]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [3]:
pip -q install langchain langchain_community langchain_openai faiss-cpu pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
from google.colab import files
num_files=int(input("how many PDF Files do you want to upload?:"))
print(f"Please select {num_files} PDF files to upload:")
uploaded=files.upload()
pdf_files=[f for f in uploaded.keys() if f.lower().endswith('.pdf')]
if len(pdf_files)!=num_files:
  print(f"Error:Expected{num_files}PDF files,but received {len(pdf_files)}")
else:
  print("\nUploaded PDF's")
  for i,pdf in enumerate(pdf_files,start=1):
    print(f"{i}.{pdf}")


how many PDF Files do you want to upload?:2
Please select 2 PDF files to upload:


Saving Reliance annual balance sheet.pdf to Reliance annual balance sheet.pdf
Saving unilever-annual-report-and-accounts-2025.pdf to unilever-annual-report-and-accounts-2025.pdf

Uploaded PDF's
1.Reliance annual balance sheet.pdf
2.unilever-annual-report-and-accounts-2025.pdf


In [5]:
from langchain_community.document_loaders import PyPDFLoader
all_documents=[]
for pdf_file_name in pdf_files:
  print(f"Loading {pdf_file_name}...")
  loader=PyPDFLoader(pdf_file_name)
  documents=loader.load()
  all_documents.extend(documents)
  print("Total Pages Loaded:",len(all_documents))
  documents=all_documents

/tmp/ipykernel_796/3799353035.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading Reliance annual balance sheet.pdf...
Total Pages Loaded: 110
Loading unilever-annual-report-and-accounts-2025.pdf...
Total Pages Loaded: 395


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=500)
docs=text_splitter.split_documents(documents)
print("Total Chunks:",len(docs))

Total Chunks: 3794


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

/tmp/ipykernel_796/4274292163.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings=HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstores=FAISS.from_documents(docs,embeddings)
print("Vector DB Created Successfully")
print(vectorstores)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB Created Successfully


In [9]:
retriever=vectorstores.as_retriever(search_kwargs={"k":5})
print(retriever)

tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7cb174952210> search_kwargs={'k': 5}


In [10]:
print(vectorstores.index_to_docstore_id)

{0: '49191838-acff-4b4b-b568-50861c64dce5', 1: '4c37359c-5352-4f7f-968e-203b8d984237', 2: 'd1256c79-923a-49a8-8b68-b8d1609e5326', 3: '3b275ae4-29ef-4d99-a5cf-20504d717393', 4: '3503a55f-ec9f-48e7-a26d-5562c1bd2863', 5: 'a6ec85a5-6215-4ed2-9ba2-b5ca0c9ed185', 6: 'bfc5b8c6-7dd5-441e-8ec3-8f24f0ff6f9f', 7: 'c7f13005-2d8e-404f-b40c-2818633f58bc', 8: '65b06269-755e-43d2-a781-fce169d95ad2', 9: 'e55ff512-8cc2-40ca-a931-0772f565929a', 10: '429273f3-0e14-434c-8530-5917c3f11f1f', 11: 'bf3e3bf1-eb62-4379-b884-cb3e2f1b959d', 12: 'd021a731-9687-43d3-8dc2-cf2d37cbd5d6', 13: '7e0d9954-d9f4-43ea-aead-466413f80644', 14: 'a8967e3c-02ae-4f46-b194-947c73cecbec', 15: 'ac36b6b9-9588-4937-940b-54ee19a31995', 16: 'bd4c0aef-a9c8-4a17-affc-b1a5c0acbe06', 17: 'eefd6885-c89f-4007-b545-6375438fb9f0', 18: '213d8ecb-951a-4bb1-a15d-f2718aa43d84', 19: 'f87e6c6c-501d-40ad-b718-9bfe7324e745', 20: 'b626a196-111d-4073-9a48-d4aacaa804fe', 21: '181b7e92-ff35-429a-a956-9a50491a4ecd', 22: '5436b9bc-bbe2-4ca2-80db-27a35b341769

In [11]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [14]:
!pip -q install flask pyngrok

In [15]:
from flask import Flask, request, jsonify, render_template_string
from pyngrok import ngrok
ngrok.kill()

In [16]:
app = Flask(__name__)

In [17]:
def financial_analyst(question):

    prompt = f"""
You are an expert AI Financial Analyst.

Rules:
- If the PDF contains tabular financial information relevant to the question,
  return the answer as a valid HTML table.
- Use bullet points for lists.
- Use paragraphs for summaries.
- Return valid HTML.

Question:
{question}
"""

    response = rag_chain.invoke(prompt)

    return str(response)

In [18]:
HTML = """
<!DOCTYPE html>
<html>
<head>

<title>AI Financial Analyst</title>

<style>

*{
    margin:0;
    padding:0;
    box-sizing:border-box;
    font-family:Arial,sans-serif;
}

body{
    display:flex;
    height:100vh;
    background:#212121;
    color:white;
}

/* Sidebar */

.sidebar{
    width:260px;
    background:#171717;
    padding:20px;
    overflow-y:auto;
    border-right:1px solid #333;
}

.sidebar h2{
    margin-bottom:20px;
}

.new-chat{
    width:100%;
    padding:12px;
    border:none;
    border-radius:10px;
    background:#19c37d;
    cursor:pointer;
    font-weight:bold;
}

.sidebar h3{
    margin-top:25px;
    margin-bottom:15px;
}

/* Chat History */

#history{
    max-height:500px;
    overflow-y:auto;
}

.history-item{
    background:#2d2d2d;
    padding:12px;
    margin-bottom:10px;
    border-radius:8px;
    cursor:pointer;
    word-wrap:break-word;
}

.history-item:hover{
    background:#3d3d3d;
}

/* Main Area */

.main{
    flex:1;
    display:flex;
    flex-direction:column;
}

.header{
    padding:20px;
    border-bottom:1px solid #333;
    font-size:24px;
    font-weight:bold;
}

.chat-container{
    flex:1;
    overflow-y:auto;
    padding:30px;
}

/* Messages */

.message{
    display:flex;
    margin-bottom:20px;
}

.user{
    justify-content:flex-end;
}

.bot{
    justify-content:flex-start;
}

.bubble{
    max-width:75%;
    padding:15px;
    border-radius:15px;
    line-height:1.6;
}

.user .bubble{
    background:#19c37d;
    color:black;
}

.bot .bubble{
    background:#2d2d2d;
    color:white;
}

/* Typing Indicator */

.typing{
    color:#bdbdbd;
    margin-bottom:15px;
}

/* Input Area */

.input-area{
    display:flex;
    gap:10px;
    padding:20px;
    border-top:1px solid #333;
}

textarea{
    flex:1;
    resize:none;
    height:55px;
    border:none;
    border-radius:12px;
    padding:15px;
    background:#2d2d2d;
    color:white;
    font-size:15px;
}

button{
    padding:0 25px;
    border:none;
    border-radius:12px;
    background:#19c37d;
    cursor:pointer;
    font-weight:bold;
}

/* TABLE STYLING */

table{
    width:100%;
    border-collapse:collapse;
    margin-top:15px;
    margin-bottom:15px;
    background:#2d2d2d;
    border-radius:12px;
    overflow:hidden;
}

th{
    background:#19c37d;
    color:black;
    padding:12px;
    text-align:left;
    font-weight:bold;
}

td{
    padding:12px;
    border-bottom:1px solid #444;
}

tr:nth-child(even){
    background:#262626;
}

tr:hover{
    background:#3a3a3a;
}

/* Bullet Lists */

ul{
    margin-left:20px;
    margin-top:10px;
}

li{
    margin-bottom:6px;
}

</style>

</head>

<body>

<div class="sidebar">

    <h2>📊 AI Financial Analyst</h2>

    <button class="new-chat"
            onclick="newChat()">
        + New Chat
    </button>

    <h3>Chat History</h3>

    <div id="history"></div>

</div>

<div class="main">

    <div class="header">
        AI Financial Analyst
    </div>

    <div class="chat-container" id="chat">

        <div class="message bot">

            <div class="bubble">

                Hello! Upload a financial report and ask me anything.

                <br><br>

                Example questions:
                <ul>
                    <li>Summarize the annual report.</li>
                    <li>Show revenue and profit trends.</li>
                    <li>List major risk factors.</li>
                    <li>Display balance sheet data.</li>
                </ul>

            </div>

        </div>

    </div>

    <div class="input-area">

        <textarea id="question"
                  placeholder="Ask anything about the financial report..."></textarea>

        <button onclick="askQuestion()">
            Send
        </button>

    </div>

</div>

<script>
/* VARIABLES */

let conversations =
JSON.parse(localStorage.getItem("conversations")) || [];

let currentConversation = [];


/* ASK QUESTION */

function askQuestion(){

    let question =
    document.getElementById("question").value.trim();

    if(question === "") return;

    let chat =
    document.getElementById("chat");

    /* Show User Message */

    chat.innerHTML += `
    <div class="message user">
        <div class="bubble">
            ${question}
        </div>
    </div>`;

    /* Typing Indicator */

    chat.innerHTML += `
    <div class="typing" id="typing">
        AI Financial Analyst is thinking...
    </div>`;

    chat.scrollTop = chat.scrollHeight;

    fetch('/ask',{

        method:'POST',

        headers:{
            'Content-Type':
            'application/x-www-form-urlencoded'
        },

        body:'question=' +
        encodeURIComponent(question)

    })

    .then(response => response.json())

    .then(data => {

        /* Remove Typing */

        let typing =
        document.getElementById("typing");

        if(typing){
            typing.remove();
        }

        /* Show AI Response */

        chat.innerHTML += `
        <div class="message bot">
            <div class="bubble">
                ${data.answer}
            </div>
        </div>`;

        /* Save Conversation */

        currentConversation.push({

            question: question,

            answer: data.answer

        });

        document.getElementById("question").value = "";

        chat.scrollTop = chat.scrollHeight;

    })

    .catch(error => {

        let typing =
        document.getElementById("typing");

        if(typing){
            typing.remove();
        }

        chat.innerHTML += `
        <div class="message bot">
            <div class="bubble">
                Error: Unable to get response.
            </div>
        </div>`;

    });

}


/* NEW CHAT */

function newChat(){

    if(currentConversation.length > 0){

        conversations.push({

            title:
            currentConversation[0].question,

            messages:
            [...currentConversation]

        });

        localStorage.setItem(

            "conversations",

            JSON.stringify(conversations)

        );
    }

    currentConversation = [];

    updateSidebar();

    document.getElementById("chat").innerHTML = `

    <div class="message bot">

        <div class="bubble">

            New chat started.

            <br><br>

            Ask me anything about your financial reports.

        </div>

    </div>`;
}


/* UPDATE SIDEBAR */

function updateSidebar(){

    let history =
    document.getElementById("history");

    history.innerHTML = "";

    conversations.forEach((chat,index)=>{

        history.innerHTML += `

        <div class="history-item"

             onclick="openChat(${index})">

            ${chat.title}

        </div>`;

    });

}


/* OPEN PREVIOUS CHAT */

function openChat(index){

    let chatBox =
    document.getElementById("chat");

    chatBox.innerHTML = "";

    conversations[index].messages.forEach(msg=>{

        chatBox.innerHTML += `

        <div class="message user">

            <div class="bubble">

                ${msg.question}

            </div>

        </div>

        <div class="message bot">

            <div class="bubble">

                ${msg.answer}

            </div>

        </div>`;
    });

    currentConversation =
    [...conversations[index].messages];

    chatBox.scrollTop =
    chatBox.scrollHeight;
}


/* SEND MESSAGE USING ENTER */

document.getElementById("question")

.addEventListener("keypress",

function(event){

    if(event.key === "Enter" && !event.shiftKey){

        event.preventDefault();

        askQuestion();
    }

});


/* LOAD HISTORY */

updateSidebar();

</script>

</body>

</html>
"""

In [19]:
@app.route('/')
def home():
    return render_template_string(HTML)


@app.route('/ask', methods=['POST'])
def ask():

    question = request.form['question']

    answer = financial_analyst(question)

    return jsonify({
        "answer": answer
    })

In [20]:
ngrok.set_auth_token( "YOUR_GROQ_API_KEY")

In [ ]:
public_url = ngrok.connect(5000)

print(public_url)

In [ ]:
app.run(
    host='0.0.0.0',
    port=5000,
    use_reloader=False
)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [20/Jun/2026 04:20:02] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Jun/2026 04:20:03] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [20/Jun/2026 04:28:57] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Jun/2026 04:29:16] "POST /ask HTTP/1.1" 200 -
